# Cascade vs Direct — Colab 실행 노트북

런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 위에서부터 순서대로 실행하세요.

In [ ]:
!nvidia-smi

## 1. 레포 받기 & 설치
`gitminseung`를 본인 GitHub 아이디로 바꾸세요.

In [ ]:
GITHUB_ID = 'gitminseung'
!git clone https://github.com/{GITHUB_ID}/cascade-vs-direct-slu.git
%cd cascade-vs-direct-slu
!pip -q install -r requirements.txt

## 2. 데이터 준비 (Speech-MASSIVE ko-KR + MASSIVE 텍스트)
약 650MB 다운로드. 평가셋(`data/eval/`)과 학습용 오디오/텍스트가 생성됩니다.

In [ ]:
!python data/prepare_data.py

## 3. Cascade 2단계: 한국어 텍스트 의도 분류기 학습 (KLUE-RoBERTa)

In [ ]:
!python baseline/train_text_clf.py

## 4. 시스템 실행

In [ ]:
!python baseline/cascade.py --oracle
!python baseline/cascade.py --asr openai/whisper-small
!python baseline/cascade.py --asr openai/whisper-large-v3-turbo

In [ ]:
!python baseline/train_direct.py --encoder openai/whisper-small
!python baseline/direct_infer.py --ckpt checkpoints/direct_whisper-small

## 5. 평가 & 리더보드

In [ ]:
!python evaluation/evaluate.py results/preds_*.jsonl
!python evaluation/make_leaderboard.py

## 6. 결과 다운로드
`results_bundle.zip`에 평가셋(`data/eval`, `data/labels.json`)과 지표·리더보드가 들어 있습니다. 압축을 풀어 레포에 커밋하고, 리더보드는 `make_leaderboard.py --html <제출 HTML 경로>`로 제출 페이지에 반영하세요.

In [ ]:
!zip -qr results_bundle.zip data/eval data/labels.json results/metrics_*.json results/leaderboard.*
from google.colab import files
files.download('results_bundle.zip')